In [11]:
import os, sys
week1_path = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0,week1_path)
# import scraper.py
from scraper import fetch_website_contents, fetch_website_links
# import environment variables module
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import json
openai = OpenAI()

In [2]:
# Load environment variables
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")
    

API key found and looks good so far!


In [8]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

# First Step: Get relevant links to the website and absolute links (replace relative links) using AI

In [3]:
link_system_prompt = """You are a helpful assistant for extracting links from a website. 
You will be given a list of links, and you should extract the ones that are relevant to sales brochures about the company,
such as links to an About page, a Products or Services page, or any page that seems to be focused on marketing the company's offerings. 
Please respond in a JSON format like below example, with a list of relevant links and their descriptions if available. 
{
    "links": [
        {
            "type": "About us page with company history and mission statement",
            "url": "https://example.com/about"
            
        },
        {
            "type": "Products page showcasing our offerings and features",
            "url": "https://example.com/products"
        }
    ]
}
"""


In [4]:
#function to extract relevant links from a website for user prompt.
def get_links_user_prompt(url):
    user_prompt = f"""Here is a list of links from the website {url}:
    Please respond in JSON format with a list of relevant links and their descriptions if available.
    Respond with a full https url for each link.
    Do not include privacy, terms of service, or any other irrelevant links."""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))

Here is a list of links from the website https://edwarddonner.com:
    Please respond in JSON format with a list of relevant links and their descriptions if available.
    Respond with a full https url for each link.
    Do not include privacy, terms of service, or any other irrelevant links.https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/20

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
        #This forces the model to output only valid JSON, preventing malformed or partially‑structured responses. 
        #Even if the system prompt asks for JSON, the model may not always comply, so JSON mode adds a guaranteed enforcement layer. This is an inference‑time parsing feature where the API validates the output and ensures it is valid JSON before returning it.
        #JSON mode guarantees valid JSON, but does not enforce a schema — for schema‑validated output, use Structured Outputs.  
    )
    result = response.choices[0].message.content
    links = json.loads(result) #parse the JSON string into a Python dictionary
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'Company homepage with overview of services and offerings',
   'url': 'https://edwarddonner.com/'},
  {'type': 'About the founder and Nebula project',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Curriculum page detailing courses and learning paths',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'Blog post about AI builder with N8N for agents and voice agents',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'Blog post: AI coder vibe coder to agentic engineer',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'Blog post: AI in production, gen AI and agentic AI on AWS at scale',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'Blog post: Connecting my courses to become an LLM expert and leader',
   'url': 'https://edwarddonner.com/2025

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling gpt-5-nano...")
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano...
Found 7 relevant links


{'links': [{'type': 'Enterprise and business solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'Pricing page for Hugging Face offerings',
   'url': 'https://huggingface.co/pricing'},
  {'type': 'Pricing page section: Endpoints',
   'url': 'https://huggingface.co/pricing#endpoints'},
  {'type': 'Pricing page section: Spaces',
   'url': 'https://huggingface.co/pricing#spaces'},
  {'type': 'Brand and brand assets page',
   'url': 'https://huggingface.co/brand'},
  {'type': 'HuggingFace hub marketing page',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'Endpoints API overview',
   'url': 'https://endpoints.huggingface.co'}]}

# Second step: Make the brochure

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [24]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano...
Found 5 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
2 days ago
•
123k
•
2.76k
moonshotai/Kimi-K2.6
Updated
3 days ago
•
376k
•
1.04k
Qwen/Qwen3.6-27B
Updated
2 days ago
•
330k
•
836
openai/privacy-filter
Updated
4 days ago
•
35.8k
•
808
deepseek-ai/DeepSeek-V4-Flash
Updated
2 days ago
•
46k
•
706
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
173
ML Intern
🤖
173
Chat with an AI ML intern for quick technical help
Running
on
Zero
MCP
2.38k
Wan2.2 14B Preview
🐌
2.38k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured

In [20]:
brochure_system_prompt = """You are a helpful assistant for creating sales brochures.
You will be given a list of links about a company and their website information. You should create a sales brochure
for the company based on the information from those links. 
The brochure should be concise and compelling, 
highlighting the key features and benefits of the company's products or services. 
Please structure the brochure with a clear headline, a brief introduction, and bullet points for key features or offerings.
"""

In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [27]:
print(get_brochure_user_prompt("Hugging Face", "https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano...
Found 9 relevant links

You are looking at a company called: Hugging Face
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
2 days ago
•
123k
•
2.76k
moonshotai/Kimi-K2.6
Updated
3 days ago
•
376k
•
1.04k
Qwen/Qwen3.6-27B
Updated
2 days ago
•
330k
•
836
openai/privacy-filter
Updated
4 days ago
•
35.8k
•
808
deepseek-ai/DeepSeek-V4-Flash
Updated
2 days ago
•
46k
•
706
Browse 2M+ models
Spaces
Running
on
C

In [29]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    brochure = response.choices[0].message.content
    display(Markdown(brochure))

In [30]:
create_brochure("Hugging Face", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano...
Found 6 relevant links


# Hugging Face: The Future of AI Collaboration

Join the world's leading AI community dedicated to building the future of machine learning. Hugging Face is the ultimate platform where AI enthusiasts, researchers, and organizations collaborate on cutting-edge models, datasets, and applications across all modalities including text, image, video, audio, and 3D.

---

## Why Choose Hugging Face?

- **Vast AI Model Repository**  
  Access and browse over 2 million pre-trained models spanning multiple AI domains.

- **Rich Collection of Datasets**  
  Explore and utilize more than 500,000 datasets to power your machine learning projects.

- **Interactive AI Spaces**  
  Discover and run live AI apps or create your own with zero setup using Spaces.

- **Collaborative Platform**  
  Build, share, and grow your machine learning portfolio while collaborating with a vibrant AI community.

- **Multi-Modal AI Support**  
  Work seamlessly across text, images, video, audio, and even 3D machine learning applications.

---

## Enterprise & Team Solutions

Scale your organization’s AI capabilities with Hugging Face’s enterprise-grade platform:

- **Robust Security & Compliance**  
  Single Sign-On (SSO), audit logs, organization-wide security policies, and granular access controls.

- **Advanced Compute Resources**  
  Access enhanced compute options including 5× more ZeroGPU quota and priority hosting.

- **Centralized Management**  
  Manage users, tokens, spending limits, and billing via an intuitive dashboard.

- **Collaboration on Private Repositories & Datasets**  
  Share and view private datasets effortlessly within your team.

- **Priority Enterprise Support**  
  Dedicated support to maximize your platform’s capabilities and accelerate your AI projects.

- **Flexible Plans**  
  Starting at $20/user/month for teams, with custom contracts available for enterprises.

---

## Pricing Plans

- **Free Access**  
  Basic account with community access to models, datasets, and Spaces.

- **PRO Account ($9/month)**  
  Enhanced personal experience with increased storage, inference credits, priority compute quota, and private dataset features.

- **Team & Enterprise Plans**  
  Tailored solutions for organizations requiring secure collaboration, advanced resources, and dedicated support.

---

## Get Started Today!

- Explore and collaborate with millions of AI models and datasets.  
- Build and showcase your AI projects in the community.  
- Accelerate your AI development with enterprise-grade tools and support.

**Join the AI revolution with Hugging Face — where the machine learning community builds the future.**

[Sign Up Now](https://huggingface.co/join) | [Explore Models & Datasets](https://huggingface.co/models) | [Learn About Enterprise Solutions](https://huggingface.co/enterprise)